# 11 — Re-clasificación zero-shot con etiquetas refinadas (v2)

Re-ejecuta BART-MNLI sobre el corpus con **etiquetas más específicas** para resolver el solapamiento semántico detectado en la v1 entre `phishing_identity` y `gov_impersonation`.

## Cambios clave respecto al notebook 09

1. **Etiquetas refinadas** — más descriptivas y semánticamente excluyentes.
2. **NO sobrescribe** el CSV del run anterior. Guarda en `scam_us_FINAL_classified_v2.csv`.
3. **Checkpoints más frecuentes** (cada 50 tweets en vez de 100) — más seguridad.
4. **Carga BERTopic ya hecho** del CSV anterior — no se reejecuta, solo reclasifica.
5. **Estimación de tiempo realista**: ~5-7 horas reales en CPU Intel.

## Antes de ejecutar

- ✅ Mac enchufado.
- ✅ `caffeinate -dimsu` corriendo en otra terminal.
- ✅ Kernel: **Python (tfm-nlp)**.
- ✅ Backup del `scam_us_FINAL_classified.csv` anterior en Drive (por si acaso).


## Verificación de entorno

In [1]:
import sys
print(f"Python: {sys.version}")

import numpy, torch, transformers, sentence_transformers, pandas
print(f"numpy:                 {numpy.__version__}")
print(f"torch:                 {torch.__version__}")
print(f"transformers:          {transformers.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"pandas:                {pandas.__version__}")
print()
print("✓ Entorno listo")


Python: 3.11.9 | packaged by conda-forge | (main, Apr 19 2024, 18:45:13) [Clang 16.0.6 ]
numpy:                 1.26.4
torch:                 2.2.0
transformers:          4.41.2
sentence-transformers: 3.4.1
pandas:                3.0.3

✓ Entorno listo


## Carga del corpus

Cargamos el corpus que ya tiene BERTopic procesado, **NO lo reejecutamos**. Solo reclasificaremos con zero-shot v2.


In [2]:
import pandas as pd
import numpy as np
import os
import re
import time

pd.set_option('display.max_colwidth', None)

# Cargamos el CSV que ya tiene BERTopic (más eficiente que partir del v8)
INPUT_CSV = "../data/raw/scam_us_v8_with_bertopic.csv"

if not os.path.exists(INPUT_CSV):
    # Si no existe el de BERTopic intermedio, usamos el v8 limpio
    INPUT_CSV = "../data/raw/scam_us_FINAL_v8.csv"
    print(f"⚠️ No se encontró el CSV de BERTopic. Usando {INPUT_CSV}")
    print(f"   (no tendrás columnas bertopic_id ni bertopic_keywords)")

df = pd.read_csv(INPUT_CSV)
print(f"Tweets cargados: {len(df)}")
print(f"Columnas: {list(df.columns)[:10]}...")

# Asegurar clean_text
URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")

def clean_for_analysis(text):
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

if "clean_text" not in df.columns or df["clean_text"].isna().all():
    df["clean_text"] = df["text"].apply(clean_for_analysis)
else:
    df["clean_text"] = df["clean_text"].fillna("").apply(lambda x: x if x else "")
    mask_empty = df["clean_text"] == ""
    df.loc[mask_empty, "clean_text"] = df.loc[mask_empty, "text"].apply(clean_for_analysis)

df = df[df["clean_text"].str.len() >= 20].reset_index(drop=True)
print(f"Tras filtrar textos cortos: {len(df)}")


Tweets cargados: 1928
Columnas: ['tweet_id', 'created_at', 'text', 'lang', 'author_id', 'username', 'name', 'user_location', 'user_verified', 'user_followers']...
Tras filtrar textos cortos: 1928


## Etiquetas refinadas (v2)

**Cambios clave:**
- `phishing` ahora especifica "private company" para distinguirlo de gov.
- `gov_impersonation` cita IRS, USPS, FedEx, toll → palabras que el modelo reconoce.
- Resto: descripciones más naturales y específicas.


In [3]:
CANDIDATE_LABELS_V2 = [
    "investment scam or cryptocurrency trading fraud",
    "romance scam or online dating fraud",
    "phishing email or scam text impersonating a private company or bank",
    "impersonation of government agency such as IRS USPS FedEx or toll authority",
    "bank fraud or unauthorized wire transfer",
    "scam involving Zelle Venmo Cash App PayPal or Apple Pay payment app",
    "Ponzi scheme or pyramid scheme",
    "fake tech support call or computer pop-up scam",
    "fake job offer or employment scam",
    "fake charity or donation request scam",
    "insurance fraud or insurance claim manipulation",
    "corporate fraud or securities market manipulation",
    "tax evasion or tax preparation fraud",
    "not related to financial fraud at all",
]

LABEL_TO_CODE_V2 = {
    "investment scam or cryptocurrency trading fraud": "investment_crypto",
    "romance scam or online dating fraud": "romance",
    "phishing email or scam text impersonating a private company or bank": "phishing_identity",
    "impersonation of government agency such as IRS USPS FedEx or toll authority": "gov_impersonation",
    "bank fraud or unauthorized wire transfer": "bank_wire",
    "scam involving Zelle Venmo Cash App PayPal or Apple Pay payment app": "payment_app",
    "Ponzi scheme or pyramid scheme": "ponzi_pyramid",
    "fake tech support call or computer pop-up scam": "tech_support",
    "fake job offer or employment scam": "employment",
    "fake charity or donation request scam": "charity",
    "insurance fraud or insurance claim manipulation": "insurance",
    "corporate fraud or securities market manipulation": "corporate",
    "tax evasion or tax preparation fraud": "tax",
    "not related to financial fraud at all": "not_related",
}

print(f"Categorías candidatas: {len(CANDIDATE_LABELS_V2)}\n")
for i, lbl in enumerate(CANDIDATE_LABELS_V2, 1):
    print(f"  {i:2d}. {lbl}")


Categorías candidatas: 14

   1. investment scam or cryptocurrency trading fraud
   2. romance scam or online dating fraud
   3. phishing email or scam text impersonating a private company or bank
   4. impersonation of government agency such as IRS USPS FedEx or toll authority
   5. bank fraud or unauthorized wire transfer
   6. scam involving Zelle Venmo Cash App PayPal or Apple Pay payment app
   7. Ponzi scheme or pyramid scheme
   8. fake tech support call or computer pop-up scam
   9. fake job offer or employment scam
  10. fake charity or donation request scam
  11. insurance fraud or insurance claim manipulation
  12. corporate fraud or securities market manipulation
  13. tax evasion or tax preparation fraud
  14. not related to financial fraud at all


## Carga de BART-MNLI

El modelo ya está descargado de la ejecución anterior, así que la carga será rápida (1-2 min).


In [4]:
print("⏳ Cargando modelo BART-MNLI (ya en caché)...")
t0 = time.time()

from transformers import pipeline
import torch

device = -1  # CPU
print(f"   Dispositivo: CPU")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

t_load = time.time() - t0
print(f"✓ Modelo cargado en {t_load/60:.1f} min")


⏳ Cargando modelo BART-MNLI (ya en caché)...
   Dispositivo: CPU
✓ Modelo cargado en 0.2 min


## Re-clasificación con etiquetas v2

⏳ **Tarda 5-7 horas en CPU Intel.** Checkpoints cada 50 tweets.

NO se sobrescribe el CSV anterior — guardamos en `scam_us_FINAL_classified_v2.csv` y `scam_us_v8_classification_v2_checkpoint.csv`.


In [5]:
texts_to_classify = df["clean_text"].tolist()
n = len(texts_to_classify)
BATCH_SIZE = 8
CHECKPOINT_EVERY = 50      # más frecuente que el run anterior

predictions = []
scores = []

print(f"⏳ Clasificando {n} tweets en {len(CANDIDATE_LABELS_V2)} categorías (v2)...")
print(f"   Lote: {BATCH_SIZE} | Checkpoint cada {CHECKPOINT_EVERY}")
print(f"   Output:           ../data/raw/scam_us_FINAL_classified_v2.csv")
print(f"   Checkpoints:      ../data/raw/scam_us_v8_classification_v2_checkpoint.csv")
print(f"   ESTIMACIÓN: 5-7 HORAS en CPU Intel. Sé paciente.\n")

t0 = time.time()
last_print = t0

for i in range(0, n, BATCH_SIZE):
    batch = texts_to_classify[i:i+BATCH_SIZE]

    try:
        results = classifier(
            batch,
            candidate_labels=CANDIDATE_LABELS_V2,
            multi_label=False,
        )
        if isinstance(results, dict):
            results = [results]
    except Exception as e:
        print(f"  ⚠️ Error en lote {i}: {e}")
        for _ in batch:
            predictions.append("not related to financial fraud at all")
            scores.append(0.0)
        continue

    for r in results:
        predictions.append(r["labels"][0])
        scores.append(r["scores"][0])

    if (time.time() - last_print) > 30 or (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - t0
        done = min(i + BATCH_SIZE, n)
        eta = (elapsed / done) * (n - done) if done > 0 else 0
        print(f"  {done}/{n} ({done/n*100:.1f}%) — {elapsed/60:.1f} min — ETA: {eta/60:.1f} min")
        last_print = time.time()

    # Checkpoint cada 50
    if (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0 and (i + BATCH_SIZE) < n:
        df_partial = df.iloc[:len(predictions)].copy()
        df_partial["predicted_label_v2"] = predictions
        df_partial["confidence_score_v2"] = scores
        df_partial.to_csv("../data/raw/scam_us_v8_classification_v2_checkpoint.csv", index=False)

t_classify = time.time() - t0
print(f"\n✓ Clasificación v2 completada en {t_classify/60:.1f} min")


⏳ Clasificando 1928 tweets en 14 categorías (v2)...
   Lote: 8 | Checkpoint cada 50
   Output:           ../data/raw/scam_us_FINAL_classified_v2.csv
   Checkpoints:      ../data/raw/scam_us_v8_classification_v2_checkpoint.csv
   ESTIMACIÓN: 5-7 HORAS en CPU Intel. Sé paciente.

  8/1928 (0.4%) — 1.6 min — ETA: 375.3 min
  16/1928 (0.8%) — 2.7 min — ETA: 318.1 min
  24/1928 (1.2%) — 4.0 min — ETA: 314.6 min
  32/1928 (1.7%) — 5.1 min — ETA: 300.0 min
  40/1928 (2.1%) — 6.2 min — ETA: 293.0 min
  48/1928 (2.5%) — 7.5 min — ETA: 292.3 min
  56/1928 (2.9%) — 8.6 min — ETA: 286.1 min
  64/1928 (3.3%) — 10.0 min — ETA: 291.3 min
  72/1928 (3.7%) — 11.5 min — ETA: 297.2 min
  80/1928 (4.1%) — 12.9 min — ETA: 298.3 min
  88/1928 (4.6%) — 14.1 min — ETA: 294.9 min
  96/1928 (5.0%) — 15.2 min — ETA: 289.2 min
  104/1928 (5.4%) — 16.4 min — ETA: 287.2 min
  112/1928 (5.8%) — 17.4 min — ETA: 282.6 min
  120/1928 (6.2%) — 18.9 min — ETA: 284.0 min
  128/1928 (6.6%) — 19.9 min — ETA: 280.5 min
  136

In [6]:
# Añadir predicciones v2 al DataFrame
df["predicted_label_v2"] = predictions[:len(df)]
df["confidence_score_v2"] = scores[:len(df)]
df["predicted_category_v2"] = df["predicted_label_v2"].map(LABEL_TO_CODE_V2)
df["is_relevant_v2"] = df["predicted_category_v2"] != "not_related"

# Distribución
print("=== DISTRIBUCIÓN DE CATEGORÍAS V2 (REFINADAS) ===\n")
counts = df["predicted_category_v2"].value_counts()
total = len(df)
for cat, n_cat in counts.items():
    pct = n_cat / total * 100
    print(f"  {cat:<22} {n_cat:>5}  ({pct:>5.1f}%)")

print(f"\nRelevantes: {df['is_relevant_v2'].sum()} / {total} ({df['is_relevant_v2'].mean()*100:.1f}%)")


=== DISTRIBUCIÓN DE CATEGORÍAS V2 (REFINADAS) ===

  phishing_identity        646  ( 33.5%)
  investment_crypto        295  ( 15.3%)
  ponzi_pyramid            290  ( 15.0%)
  employment               199  ( 10.3%)
  romance                  122  (  6.3%)
  bank_wire                102  (  5.3%)
  gov_impersonation         97  (  5.0%)
  tech_support              54  (  2.8%)
  charity                   48  (  2.5%)
  insurance                 28  (  1.5%)
  corporate                 23  (  1.2%)
  not_related               12  (  0.6%)
  tax                        7  (  0.4%)
  payment_app                5  (  0.3%)

Relevantes: 1916 / 1928 (99.4%)


In [7]:
# Confianza
print("=== CONFIANZA V2 ===\n")
print(f"  Media:    {df['confidence_score_v2'].mean():.3f}")
print(f"  Mediana:  {df['confidence_score_v2'].median():.3f}")
print(f"  Min:      {df['confidence_score_v2'].min():.3f}")
print(f"  Max:      {df['confidence_score_v2'].max():.3f}")
print(f"\n  Alta confianza (≥0.7):  {(df['confidence_score_v2'] >= 0.7).sum()}")
print(f"  Media (0.4-0.7):        {((df['confidence_score_v2'] >= 0.4) & (df['confidence_score_v2'] < 0.7)).sum()}")
print(f"  Baja (<0.4):            {(df['confidence_score_v2'] < 0.4).sum()}")


=== CONFIANZA V2 ===

  Media:    0.474
  Mediana:  0.442
  Min:      0.100
  Max:      0.987

  Alta confianza (≥0.7):  381
  Media (0.4-0.7):        696
  Baja (<0.4):            851


## Comparativa v1 vs v2

Si el CSV del run anterior existe, comparamos cuánto cambia la distribución.


In [8]:
V1_PATH = "../data/raw/scam_us_FINAL_classified.csv"
if os.path.exists(V1_PATH):
    df_v1 = pd.read_csv(V1_PATH)
    if "predicted_category" in df_v1.columns:
        # Mergear por tweet_id
        df_compare = df.merge(
            df_v1[["tweet_id", "predicted_category"]].rename(
                columns={"predicted_category": "predicted_category_v1"}
            ),
            on="tweet_id",
            how="left",
        )
        
        print("=== COMPARATIVA V1 vs V2 (en mismos tweets) ===\n")
        # Distribución lado a lado
        v1_counts = df_compare["predicted_category_v1"].value_counts()
        v2_counts = df_compare["predicted_category_v2"].value_counts()
        all_cats = sorted(set(v1_counts.index) | set(v2_counts.index))
        print(f"{'categoría':<22} {'v1':>6} {'v2':>6} {'cambio':>8}")
        print("-" * 50)
        for cat in all_cats:
            n1 = v1_counts.get(cat, 0)
            n2 = v2_counts.get(cat, 0)
            diff = n2 - n1
            sign = "+" if diff > 0 else ""
            print(f"  {cat:<20} {n1:>6} {n2:>6} {sign}{diff:>6}")
        
        # Cuántos tweets cambiaron de categoría
        n_changed = (df_compare["predicted_category_v1"] != df_compare["predicted_category_v2"]).sum()
        print(f"\nTweets que cambiaron de categoría: {n_changed} ({n_changed/len(df_compare)*100:.1f}%)")
    else:
        print("V1 no tiene columna predicted_category")
else:
    print("No existe el CSV v1 para comparar")


=== COMPARATIVA V1 vs V2 (en mismos tweets) ===

categoría                  v1     v2   cambio
--------------------------------------------------
  bank_wire                59    106 +    47
  charity                  34     48 +    14
  corporate                20     23 +     3
  employment              248    199    -49
  gov_impersonation         5     99 +    94
  insurance                 5     28 +    23
  investment_crypto       297    297      0
  not_related              36     12    -24
  payment_app              37      5    -32
  phishing_identity       642    652 +    10
  ponzi_pyramid           302    294     -8
  romance                 198    124    -74
  tax                      22      7    -15
  tech_support             43     54 +    11

Tweets que cambiaron de categoría: 446 (22.9%)


## Guardado del CSV v2

NO sobrescribimos el v1. El v2 va a un archivo separado.


In [9]:
OUTPUT_V2 = "../data/raw/scam_us_FINAL_classified_v2.csv"
df.to_csv(OUTPUT_V2, index=False, encoding="utf-8")

print(f"✓ Guardado: {OUTPUT_V2}")
print(f"  Total filas: {len(df)}")
print()
print("📦 Archivos:")
print(f"  scam_us_FINAL_classified.csv      ← run v1 (original, intacto)")
print(f"  scam_us_FINAL_classified_v2.csv   ← run v2 (este, refinado)")
print()
print("🚨 HAZ COPIA DE SEGURIDAD del v2 a Drive.")


✓ Guardado: ../data/raw/scam_us_FINAL_classified_v2.csv
  Total filas: 1928

📦 Archivos:
  scam_us_FINAL_classified.csv      ← run v1 (original, intacto)
  scam_us_FINAL_classified_v2.csv   ← run v2 (este, refinado)

🚨 HAZ COPIA DE SEGURIDAD del v2 a Drive.


## Ejemplos de la nueva clasificación

Verifica que ahora `gov_impersonation` recibe más tweets de IRS/USPS/toll/FedEx.


In [10]:
print("=== 3 EJEMPLOS DE CADA CATEGORÍA V2 (alta confianza) ===\n")
for cat_code in df["predicted_category_v2"].value_counts().index:
    subset = df[
        (df["predicted_category_v2"] == cat_code) &
        (df["confidence_score_v2"] >= 0.5)
    ].nlargest(3, "confidence_score_v2")
    n_total = (df["predicted_category_v2"] == cat_code).sum()
    print(f"--- {cat_code} ({n_total} total) ---")
    for _, row in subset.iterrows():
        print(f"  [{row['confidence_score_v2']:.2f}] {str(row['text'])[:200]}")
    print()


=== 3 EJEMPLOS DE CADA CATEGORÍA V2 (alta confianza) ===

--- phishing_identity (646 total) ---
  [0.97] Received a text like this recently?

A common 'smishing' scam is when criminals send messages pretending to be your bank.

Remember to contact your bank directly if you'd like to get in touch with the
  [0.96] This is awesome. I just got a phishing email trying to get me to use my logon credentials at a financial institution. Guess what domain they hacked for this phishing scheme? https://t.co/iP85DsAWdg - 
  [0.96] SCAM ALERT! A new PayPal scam is circulating, with fake emails, texts, and calls asking for your login details. Never share your credentials with anyone, and if you’re unsure, always double-check by c

--- investment_crypto (295 total) ---
  [0.97] The "Pig Butchering" Scam: When Romance Meets Fake Crypto Investing
#PigButchering #pigbutcheringscam #fakecrypto
https://t.co/tacWHnT91E https://t.co/vnh6nU7URO
  [0.97] Crypto Investment Scam Reported in Ashburn https://t.co

In [11]:
# Específicamente: ver ejemplos de gov_impersonation
print("=== EJEMPLOS DE gov_impersonation EN V2 ===\n")
gov_subset = df[df["predicted_category_v2"] == "gov_impersonation"].nlargest(
    10, "confidence_score_v2"
)
for _, row in gov_subset.iterrows():
    print(f"  [{row['confidence_score_v2']:.2f}] @{row['username']}")
    print(f"    {str(row['text'])[:250]}")
    print()


=== EJEMPLOS DE gov_impersonation EN V2 ===

  [0.59] @CCarol36410453
    Beware of fake text messages from the BMV in your state saying you owe money.  I called the BMV and the toll and they don't send text messages about money owed.

  [0.54] @exploreclarion
    State police are investigating the theft of scrap metal valued at over $6,000, a fraudulent IRS letter demanding thousands of dollars, and a drug possession case at a Clarion County school.
https://t.co/Wdu1oddqTD

  [0.51] @M00nlit0rchids_
    A tip as to How to Handle those ridiculous IRS scam callers … 😀😅😂🤣 https://t.co/bDB7FlWbRA via @YouTube

  [0.50] @Pinky_Balboaaa
    The tollway scam text was a little tempting because I actually did owe them mfs some money… I was like 😰

  [0.48] @MyFDOT
    Smishing scams see an increase during the holiday season, and we’re here to remind you that FDOT will NEVER send a text message about toll invoices or balances. 

 If you receive a text message like the one above 

DON’T:  

- Cl